# <font color="#418FDE" size="6.5" uppercase>**Lasso Elastic Net**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Fit Lasso and Elastic Net models and interpret sparse coefficients. 
- Use cross-validation to select alpha and l1_ratio values. 
- Analyze convergence warnings, coefficient paths, and sparsity-based feature selection. 


## **1. Sparse Regression**

### **1.1. Lasso Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_01_01.jpg?v=1783992396" width="250">



>* Lasso penalizes unnecessary predictor coefficients
>* Zero coefficients create simpler sparse models

>* Nonzero coefficients show retained useful predictors
>* Standardization makes coefficient sizes easier to compare

>* Simplifies large predictor sets for clearer models
>* Interpret sparse choices cautiously with domain knowledge



In [ ]:
#@title Python Code - Lasso Regression

# This example fits a sparse Lasso model.
# Standardization makes coefficient sizes easier to compare.
# Zero coefficients show automatic feature selection.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_regression

from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create data where only three features truly matter.
features, target, true_coefficients = make_regression(
    n_samples=160,
    n_features=8,
    n_informative=3,
    coef=True,
    noise=12.0,
    random_state=42,
)

# Validate the generated regression table shape.
if features.shape != (160, 8):
    raise ValueError("Expected 160 rows and 8 feature columns.")

feature_names = ["feature_" + str(number) for number in range(1, 9)]

# Split data before scaling to avoid leakage.
train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# Lasso uses alpha to control coefficient shrinkage.
model = make_pipeline(
    StandardScaler(),
    Lasso(alpha=8.0, max_iter=10000, random_state=42),
)

model.fit(train_features, train_target)

lasso_step = model.named_steps["lasso"]
coefficients = lasso_step.coef_

# Summarize which standardized coefficients survived.
summary = pd.DataFrame(
    {"feature": feature_names, "coef": np.round(coefficients, 2)}
)

summary["selected"] = summary["coef"] != 0
selected_count = int(summary["selected"].sum())
test_score = model.score(test_features, test_target)

print("scikit-learn version:", sklearn.__version__)
print("Test R^2:", round(test_score, 3))
print("Selected features:", selected_count, "of", len(feature_names))
print(summary.sort_values("coef", key=np.abs, ascending=False).to_string(index=False))



### **1.2. Elastic Net**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_01_02.jpg?v=1783992398" width="250">



>* Selects important predictors with zeroed coefficients
>* Handles correlated features more stably than Lasso

>* Zero coefficients remove unhelpful features
>* Shrinkage keeps related predictors stable

>* Handles many correlated predictors with balanced sparsity
>* Interpret coefficients carefully, not causally



In [ ]:
#@title Python Code - Elastic Net

# This example fits an Elastic Net model.
# Sparse coefficients show automatic feature selection.
# Cross-validation chooses regularization strength automatically.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_regression

from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create data with many noisy predictors.
features, target, true_coefficients = make_regression(
    n_samples=220,
    n_features=12,
    n_informative=4,
    coef=True,
    noise=18.0,
    random_state=42,
)

# Confirm the generated table has the expected shape.
if features.shape != (220, 12):
    raise ValueError("Expected 220 rows and 12 features.")

feature_names = []
for index in range(features.shape[1]):
    feature_names.append("feature_" + str(index + 1))

# Split data before scaling to avoid leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# ElasticNetCV tests alpha values using cross-validation.
model = make_pipeline(
    StandardScaler(),
    ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8], cv=5, max_iter=10000),
)

model.fit(X_train, y_train)
score = model.score(X_test, y_test)
elastic_net = model.named_steps["elasticnetcv"]

# Put coefficients beside feature names for interpretation.
coefficient_table = pd.DataFrame(
    {"feature": feature_names, "coefficient": elastic_net.coef_}
)

coefficient_table["absolute"] = coefficient_table["coefficient"].abs()
coefficient_table = coefficient_table.sort_values("absolute", ascending=False)
active_count = int(np.sum(elastic_net.coef_ != 0.0))

print("scikit-learn version:", sklearn.__version__)
print("Chosen alpha:", round(float(elastic_net.alpha_), 3))
print("Chosen l1_ratio:", round(float(elastic_net.l1_ratio_), 2))
print("Test R^2:", round(float(score), 3))
print("Nonzero coefficients:", active_count, "of", len(feature_names))
print(coefficient_table.head(5).round(2).to_string(index=False))



### **1.3. Sparsity Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_01_03.jpg?v=1783992400" width="250">



>* Keeps only predictors with useful signal
>* Zero coefficients improve clarity and robustness

>* Nonzero coefficients show retained predictive features
>* Zero coefficients lack unique selected value

>* Lasso sparsifies; Elastic Net keeps correlated groups
>* Interpret selected features within modeling choices



In [ ]:
#@title Python Code - Sparsity Selection

# This example demonstrates sparse regression feature selection.
# Lasso keeps useful predictors and zeros others.
# The printed coefficients reveal selected features.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create data with many features but few true signals.
features, target, true_coefficients = make_regression(
    n_samples=160,
    n_features=10,
    n_informative=3,
    coef=True,
    noise=12.0,
    random_state=42,
)

# Give each column a beginner-friendly feature name.
feature_names = ["feature_" + str(number) for number in range(1, 11)]

# Check that the generated data has the expected shape.
if features.shape != (160, 10):
    raise ValueError("Expected 160 rows and 10 feature columns.")

# Split data before scaling to avoid data leakage.
train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# LassoCV chooses alpha using cross-validation on training data.
model = make_pipeline(
    StandardScaler(),
    LassoCV(cv=5, random_state=42, max_iter=10000),
)

model.fit(train_features, train_target)

# Extract the fitted Lasso model from the pipeline.
lasso_model = model.named_steps["lassocv"]
coefficients = lasso_model.coef_

# Count coefficients that survived the Lasso penalty.
selected_count = int(np.sum(coefficients != 0))
test_score = model.score(test_features, test_target)

# Compare estimated sparse coefficients with true signals.
summary = pd.DataFrame(
    {"feature": feature_names, "lasso_coef": coefficients, "true_coef": true_coefficients}
)

summary["abs_lasso_coef"] = summary["lasso_coef"].abs()
summary = summary.sort_values("abs_lasso_coef", ascending=False)
summary = summary.drop(columns="abs_lasso_coef").head(5).round(2)

print("scikit-learn version:", sklearn.__version__)
print("Chosen alpha:", round(float(lasso_model.alpha_), 3))
print("Nonzero coefficients:", selected_count, "out of", len(feature_names))
print("Test R-squared:", round(float(test_score), 3))
print(summary.to_string(index=False))



## **2. Cross Validated Models**

### **2.1. Lasso Cross Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_02_01.jpg?v=1783992390" width="250">



>* Choose Lasso regularization using cross validation
>* Balance coefficient sparsity with prediction accuracy

>* Choose alpha using average validation performance
>* Balance sparse features with predictive accuracy

>* Chosen alpha balances accuracy and sparsity
>* Prefer simpler models when performance is similar



In [ ]:
#@title Python Code - Lasso Cross Validation

# Demonstrate Lasso cross validation.
# Choose alpha using validation folds.
# Show sparsity and test accuracy.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create regression data with only a few useful features.
features, target, true_coefficients = make_regression(
    n_samples=220,
    n_features=12,
    n_informative=4,
    noise=18.0,
    coef=True,
    random_state=42,
)

# Check that the generated data has the expected shape.
if features.shape != (220, 12):
    raise ValueError("Expected 220 rows and 12 feature columns.")

# Split once for final testing after cross validation.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# Try several alpha values inside five-fold cross validation.
alpha_grid = np.logspace(-2, 1, 25)
lasso_cv = LassoCV(
    alphas=alpha_grid,
    cv=5,
    max_iter=10000,
    random_state=42,
)

# Standardization keeps the penalty fair across features.
model = make_pipeline(StandardScaler(), lasso_cv)
model.fit(X_train, y_train)

# Evaluate the selected model on unseen test data.
predictions = model.predict(X_test)
test_rmse = mean_squared_error(y_test, predictions) ** 0.5

# Count how many coefficients remain after Lasso shrinkage.
selected_lasso = model.named_steps["lassocv"]
nonzero_count = int(np.sum(selected_lasso.coef_ != 0))

print("scikit-learn version:", sklearn.__version__)
print("Best alpha from 5-fold CV:", round(float(selected_lasso.alpha_), 4))
print("Nonzero coefficients kept:", nonzero_count, "of", features.shape[1])
print("Test RMSE:", round(float(test_rmse), 2))
print("First five coefficients:", np.round(selected_lasso.coef_[:5], 2))



### **2.2. Elastic Net Cross Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_02_02.jpg?v=1783992392" width="250">



>* Tune penalty strength and Lasso-Ridge balance
>* Validate prediction with correlated feature groups

>* Tune penalty strength and Lasso-Ridge balance
>* Cross-validation selects predictive, stable feature patterns

>* Compare nearby tuning values, not just winners
>* Scale features before selecting balanced models



In [ ]:
#@title Python Code - Elastic Net Cross Validation

# Elastic Net cross validation selects penalty settings.
# Scaling keeps coefficient penalties fairly comparable.
# The output shows accuracy and selected sparsity.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create correlated regression data with only few useful features.
features, target, true_coefficients = make_regression(
    n_samples=220,
    n_features=12,
    n_informative=4,
    coef=True,
    noise=18.0,
    random_state=42,
)

# Add correlation so Elastic Net has a realistic challenge.
features[:, 8] = features[:, 0] * 0.85 + features[:, 8] * 0.15
features[:, 9] = features[:, 1] * 0.80 + features[:, 9] * 0.20

# Check the generated table has the expected beginner-sized shape.
if features.shape != (220, 12):
    raise ValueError("Expected 220 rows and 12 feature columns.")

# Split once, then cross validation happens only inside training data.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# Try a small grid of penalty strengths and Lasso Ridge mixes.
alphas = np.logspace(-2, 1, 12)
l1_ratios = [0.2, 0.5, 0.8, 0.95]

# StandardScaler prevents feature scale from controlling the penalty.
model = make_pipeline(
    StandardScaler(),
    ElasticNetCV(
        alphas=alphas,
        l1_ratio=l1_ratios,
        cv=5,
        max_iter=10000,
        random_state=42,
    ),
)

# Fit the cross validated Elastic Net model.
model.fit(X_train, y_train)

# Evaluate the selected model on data not used for fitting.
predictions = model.predict(X_test)
test_rmse = mean_squared_error(y_test, predictions) ** 0.5

# Count nonzero coefficients to show sparsity-based feature selection.
elastic_net = model.named_steps["elasticnetcv"]
nonzero_count = int(np.sum(np.abs(elastic_net.coef_) > 1e-8))

print("scikit-learn version:", sklearn.__version__)
print("Selected alpha:", round(float(elastic_net.alpha_), 4))
print("Selected l1_ratio:", round(float(elastic_net.l1_ratio_), 2))
print("Test RMSE:", round(float(test_rmse), 2))
print("Nonzero coefficients:", nonzero_count, "of", features.shape[1])



### **2.3. Multitask Cross Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_02_03.jpg?v=1783992394" width="250">



>* Model related outcomes with shared predictors
>* Cross-validate regularization for simpler generalization

>* Validate predictions jointly across all outcomes
>* Choose regularization balancing accuracy and sparsity

>* Evaluate accuracy and shared feature selection
>* Choose stable predictors across related outcomes



In [ ]:
#@title Python Code - Multitask Cross Validation

# This example tunes multitask regularization with cross-validation.
# Related targets share a sparse predictor pattern.
# The output shows selected alpha and shared features.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import MultiTaskLassoCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create related regression targets with only a few useful features.
features, targets, true_coefs = make_regression(
    n_samples=180,
    n_features=12,
    n_informative=4,
    n_targets=3,
    noise=12.0,
    coef=True,
    random_state=42,
)

# Check that the generated data has the expected multitask shape.
if targets.ndim != 2 or targets.shape[1] != 3:
    raise ValueError("Expected three regression targets for multitask learning.")

# Split first, so scaling learns only from the training rows.
train_x, test_x, train_y, test_y = train_test_split(
    features,
    targets,
    test_size=0.25,
    random_state=42,
)

# Standardization makes one alpha value comparable across features.
scaler = StandardScaler()
train_x_scaled = scaler.fit_transform(train_x)
test_x_scaled = scaler.transform(test_x)

# Cross-validation chooses alpha for all targets jointly.
alpha_grid = np.logspace(-2, 1, 20)
model = MultiTaskLassoCV(
    alphas=alpha_grid,
    cv=5,
    max_iter=5000,
    random_state=42,
)

# Fit one multitask model instead of three separate models.
model.fit(train_x_scaled, train_y)
predictions = model.predict(test_x_scaled)

# A feature is selected when any target keeps its coefficient.
selected_mask = np.any(np.abs(model.coef_) > 1e-6, axis=0)
selected_features = np.where(selected_mask)[0]
test_rmse = mean_squared_error(test_y, predictions) ** 0.5

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Cross-validated alpha: {model.alpha_:.3f}")
print(f"Test RMSE across all targets: {test_rmse:.2f}")
print(f"Selected shared feature indices: {selected_features.tolist()}")
print(f"Number of selected features: {selected_mask.sum()} of {features.shape[1]}")



## **3. Optimization Diagnostics**

### **3.1. Coordinate Descent**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_03_01.jpg?v=1783992387" width="250">



>* Updates one coefficient at a time
>* Shrinks coefficients until the model stabilizes

>* Scaling and correlations affect convergence speed
>* Warnings require cautious coefficient interpretation

>* Regularization drives sparse feature selection
>* Confirm convergence before trusting selected features



### **3.2. Convergence Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_03_02.jpg?v=1783992385" width="250">



>* Warnings mean coefficients may not be stable
>* Selected features may change after better optimization

>* Standardize predictors to improve convergence
>* Tune iterations, tolerance, and regularization

>* Judge warnings by analysis goal
>* Check feature stability after diagnostic fixes



In [ ]:
#@title Python Code - Convergence Checks

# This script checks Elastic Net convergence behavior.
# Scaling helps coordinate descent settle reliably.
# Stable active features increase interpretation confidence.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create correlated regression data with uneven feature scales.
features, target, true_coef = make_regression(
    n_samples=180,
    n_features=8,
    n_informative=3,
    coef=True,
    noise=12.0,
    random_state=42,
)

scale_factors = np.array([1, 10, 100, 1000, 0.1, 0.01, 50, 500])
features_unscaled = features * scale_factors

# Validate the generated data before fitting models.
if features_unscaled.shape != (180, 8):
    raise ValueError("Expected 180 rows and 8 feature columns.")

# Fit the same Elastic Net with and without standardization.
plain_model = ElasticNet(
    alpha=0.08,
    l1_ratio=0.7,
    max_iter=2000,
    tol=0.0001,
    random_state=42,
)
plain_model.fit(features_unscaled, target)

scaled_model = make_pipeline(
    StandardScaler(),
    ElasticNet(
        alpha=0.08,
        l1_ratio=0.7,
        max_iter=2000,
        tol=0.0001,
        random_state=42,
    ),
)
scaled_model.fit(features_unscaled, target)

# Compare iteration counts and selected nonzero coefficients.
scaled_elastic_net = scaled_model.named_steps["elasticnet"]
plain_active = int(np.count_nonzero(np.abs(plain_model.coef_) > 0.001))
scaled_active = int(np.count_nonzero(np.abs(scaled_elastic_net.coef_) > 0.001))

print("scikit-learn version:", sklearn.__version__)
print("Unscaled iterations:", int(plain_model.n_iter_))
print("Scaled iterations:", int(scaled_elastic_net.n_iter_))
print("Unscaled active features:", plain_active)
print("Scaled active features:", scaled_active)
print("Scaled coefficients:", np.round(scaled_elastic_net.coef_, 2))



### **3.3. Coefficient Paths**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_03_03.jpg?v=1783992389" width="250">



>* Track coefficients as regularization strength changes
>* See feature entry, shrinkage, and stability

>* Early coefficients suggest stronger feature signals
>* Correlated features complicate importance interpretation

>* Spot unstable paths and scaling issues
>* Balance sparsity, accuracy, and interpretability



In [ ]:
#@title Python Code - Coefficient Paths

# This script visualizes Lasso coefficient paths.
# Regularization strength controls sparsity and feature entry.
# The plot shows coefficients becoming nonzero.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import lasso_path
from sklearn.preprocessing import StandardScaler

# Generate a small regression dataset with only three true signals.
features, target, true_coefficients = make_regression(
    n_samples=120,
    n_features=8,
    n_informative=3,
    coef=True,
    noise=12.0,
    random_state=42,
)

# Validate the generated data before modeling.
if features.shape != (120, 8):
    raise ValueError("Expected 120 rows and 8 feature columns.")

# Standardization makes coefficient paths comparable across features.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
centered_target = target - np.mean(target)

# Larger alpha means stronger shrinkage toward zero.
alphas = np.logspace(1.5, -1.0, 60)
path_alphas, path_coefficients, _ = lasso_path(
    scaled_features,
    centered_target,
    alphas=alphas,
    max_iter=5000,
)

# Count how many coefficients are nonzero at each alpha.
nonzero_counts = np.sum(np.abs(path_coefficients) > 0.001, axis=0)
strong_alpha_count = int(nonzero_counts[0])
weak_alpha_count = int(nonzero_counts[-1])

print("scikit-learn version:", sklearn.__version__)
print("Nonzero coefficients at strongest alpha:", strong_alpha_count)
print("Nonzero coefficients at weakest alpha:", weak_alpha_count)

# Plot each feature coefficient across the alpha path.
fig, ax = plt.subplots(figsize=(8, 5))
for feature_index in range(path_coefficients.shape[0]):
    label = "feature " + str(feature_index)
    ax.plot(path_alphas, path_coefficients[feature_index], label=label)

# A log scale spreads out the regularization strengths clearly.
ax.set_xscale("log")
ax.invert_xaxis()
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Lasso Coefficient Paths")

ax.set_xlabel("alpha, stronger regularization on the left")
ax.set_ylabel("standardized coefficient value")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Lasso Elastic Net**</font>


In this lecture, you learned to:
- Fit Lasso and Elastic Net models and interpret sparse coefficients. 
- Use cross-validation to select alpha and l1_ratio values. 
- Analyze convergence warnings, coefficient paths, and sparsity-based feature selection. 

In the next Module (Module 10), we will go over 'Advanced Linear'